In [ ]:
# Clone repo + install deps
import os
if not os.path.exists('/content/CD-MSC'):
    !git clone https://github.com/saha23s/CD-MSC.git /content/CD-MSC
%cd /content/CD-MSC
!git checkout aaron/preprocessing
!git pull origin aaron/preprocessing
!pip install -q -r requirements.txt

In [ ]:
!pip install -q --force-reinstall --no-cache-dir "numpy>=1.26,<2.3" "scipy>=1.14,<1.16"

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# List available checkpoints on Drive
import pathlib
DRIVE_OUTPUTS = pathlib.Path('/content/drive/MyDrive/CD-MSC-outputs')

print('Available experiments on Drive:')
for p in sorted(DRIVE_OUTPUTS.iterdir()):
    if not p.is_dir():
        continue
    has_best  = (p / 'model' / 'model_best.pth').exists()
    has_final = (p / 'model' / 'model_final.pth').exists()
    has_cfg   = (p / 'resolved_config.json').exists()
    flags = ' '.join(filter(None, [
        'best'  if has_best  else '',
        'final' if has_final else '',
        'cfg'   if has_cfg   else '',
    ]))
    print(f'  [{flags:18s}] {p.name}')

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# Set EXP_NAME to the experiment folder shown above.
# Exp 4 (C-DANN + balanced) is our best: BAunseen=0.2626

# To try the FDA checkpoint later, switch EXP_NAME:
# EXP_NAME = 'MTRCNN_seed42_B64_E100_earlystop_min20_pati10_dann0.3_cdann_balanced_fda0.05'

EXP_NAME = 'MTRCNN_seed42_B64_E100_earlystop_min10_pati5_dann0.3_cdann_balanced'

CHECKPOINT_FILE = 'model_best.pth'   # or 'model_final.pth'
# ──────────────────────────────────────────────────────────────────────────────

EXP_DIR    = DRIVE_OUTPUTS / EXP_NAME
CHECKPOINT = EXP_DIR / 'model' / CHECKPOINT_FILE
CFG_PATH   = EXP_DIR / 'resolved_config.json'

assert EXP_DIR.exists(),    f'Not found: {EXP_DIR}'
assert CHECKPOINT.exists(), f'Checkpoint missing: {CHECKPOINT}'
assert CFG_PATH.exists(),   f'resolved_config.json missing: {CFG_PATH}'

print(f'Experiment : {EXP_NAME}')
print(f'Checkpoint : {CHECKPOINT_FILE}')
print(f'Checkpoint path: {CHECKPOINT}')
print(f'Config path    : {CFG_PATH}')

In [ ]:
from pathlib import Path
import zipfile
import shutil

# ====== EDIT THIS PATH IF NEEDED ======
EVAL_ZIP = Path("/content/drive/MyDrive/Evaluation_data.zip")
# =====================================

# Extract to Colab local storage, not GitHub repo
EXTRACT_ROOT = Path("/content/cdmsc_eval_data")
EVAL_DIR = EXTRACT_ROOT / "Evaluation_data"

# Fresh extraction
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(EVAL_ZIP, "r") as z:
    real_wavs = [
        name for name in z.namelist()
        if name.startswith("Evaluation_data/")
        and name.lower().endswith(".wav")
        and "__MACOSX" not in name
        and not Path(name).name.startswith("._")
    ]

    print("Real wav files in zip:", len(real_wavs))
    print("First 5:")
    for name in real_wavs[:5]:
        print(" ", name)

    for name in real_wavs:
        z.extract(name, EXTRACT_ROOT)

audio_files = sorted(EVAL_DIR.glob("*.wav"))

print("\nExtracted wav files:", len(audio_files))
print("Evaluation folder:", EVAL_DIR)
print("First 5 extracted paths:")
for p in audio_files[:5]:
    print(" ", p)

# Submission file_id comes from filename without .wav
print("\nExample file_id:", audio_files[0].stem)

In [ ]:
from pathlib import Path
import shutil

STATS_SRC = Path("/content/drive/MyDrive/CD-MSC-feature/training_feature_stats.json")
STATS_DST = Path("/content/CD-MSC/Development_data/feature/training_feature_stats.json")

STATS_DST.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(STATS_SRC, STATS_DST)

print("Copied:", STATS_DST)

In [ ]:
# Load config/model/checkpoint once, using predict.py logic

import json
import numpy as np
import torch
from tqdm.auto import tqdm

from framework.acoustic_feature import LogMelSpectrogram, extract_log_mel_feature
from framework.config import config_signature, feature_signature_payload, load_config
from framework.dataset import load_feature_stats, validate_feature_stats_payload
from framework.metadata import SPECIES_NAMES
from framework.utilization import build_model, choose_device, training_stats_path

config = load_config(CFG_PATH)
device = choose_device(config["device"])

extractor = LogMelSpectrogram(
    sample_rate=config["sample_rate"],
    n_fft=config["n_fft"],
    hop_length=config["hop_length"],
    win_length=config["win_length"],
    n_mels=config["n_mels"],
    fmin=config["fmin"],
    fmax=config["fmax"],
).to(device)
extractor.eval()

expected_training_stats_signature = config_signature(feature_signature_payload(config, "training"))
validate_feature_stats_payload(training_stats_path(config), expected_training_stats_signature)

if config["normalize_features"]:
    feature_mean, feature_std = load_feature_stats(training_stats_path(config))

model = build_model(config, device)
checkpoint = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded:", CHECKPOINT)
print("Device:", device)

In [ ]:
# Cell 7 — test prediction on one evaluation wav

def predict_one_audio(audio_path):
    feature = extract_log_mel_feature(
        audio_path=audio_path,
        extractor=extractor,
        sample_rate=config["sample_rate"],
        normalize_waveform=config["normalize_waveform"],
        device=device,
    )

    if config["normalize_features"]:
        feature = (feature - feature_mean) / np.maximum(feature_std, 1e-6)

    features = torch.tensor(feature, dtype=torch.float32, device=device).unsqueeze(0)
    lengths = torch.tensor([feature.shape[0]], dtype=torch.long, device=device)

    # Needed for C-DANN checkpoints.
    # This only satisfies the domain head shape; species prediction does not use this label.
    dummy_species_labels = torch.zeros(features.size(0), dtype=torch.long, device=device)

    with torch.no_grad():
        outputs = model(features, lengths, species_labels=dummy_species_labels)
        probs = torch.softmax(outputs["species_logits"], dim=1)[0]
        pred_index = int(torch.argmax(probs).item())

    return {
        "file_id": audio_path.stem,
        "predicted_species_index": pred_index,
        "predicted_species_id": pred_index + 1,
        "predicted_species": SPECIES_NAMES[pred_index],
        "probabilities": probs.detach().cpu().tolist(),
    }


test_audio = audio_files[0]
test_pred = predict_one_audio(test_audio)

print("Audio:", test_audio)
print(json.dumps({
    "file_id": test_pred["file_id"],
    "predicted_species_index": test_pred["predicted_species_index"],
    "predicted_species_id": test_pred["predicted_species_id"],
    "predicted_species": test_pred["predicted_species"],
}, indent=2))

In [ ]:
# Cell 8 — run all evaluation wavs and write submission files

from pathlib import Path
import json

from framework.metadata import SPECIES_ID_TO_NAME

SPECIES_IDS = list(SPECIES_ID_TO_NAME.keys())

OUTPUT_DIR = Path("/content/drive/MyDrive/CD-MSC-submissions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUBMISSION_PATH = OUTPUT_DIR / f"{EXP_NAME}_{CHECKPOINT_FILE.replace('.pth', '')}_submission.txt"
PROBS_PATH = OUTPUT_DIR / f"{EXP_NAME}_{CHECKPOINT_FILE.replace('.pth', '')}_probabilities.jsonl"

rows = []

with open(SUBMISSION_PATH, "w", encoding="utf-8") as sub_f, open(PROBS_PATH, "w", encoding="utf-8") as prob_f:
    sub_f.write("file_id,predicted_species_id\n")

    for audio_path in tqdm(audio_files):
        pred = predict_one_audio(audio_path)

        species_id = int(SPECIES_IDS[pred["predicted_species_index"]])

        sub_f.write(f'{pred["file_id"]},{species_id}\n')

        prob_f.write(json.dumps({
            "file_id": pred["file_id"],
            "predicted_species_index": pred["predicted_species_index"],
            "predicted_species_id": species_id,
            "predicted_species": pred["predicted_species"],
            "probabilities": pred["probabilities"],
        }) + "\n")

        rows.append((pred["file_id"], species_id))

print("Wrote submission:", SUBMISSION_PATH)
print("Wrote probabilities:", PROBS_PATH)
print("Rows:", len(rows))
print("First 5:", rows[:5])